# RuleShift Benchmark

Use this notebook to run the official RuleShift benchmark task inside Kaggle.

| What this notebook does | Why it matters |
| --- | --- |
| Loads the frozen benchmark rows | Ensures every submission is evaluated on the same prompts |
| Normalizes model outputs | Makes scoring robust to a few response formats |
| Scores every episode | Produces the final leaderboard numerator and denominator |
| Registers the Kaggle task | Exposes the benchmark through the expected entry point |

---

> Recommended flow: run the notebook top to bottom once, inspect the smoke-check result, then use the registered task as the official benchmark entry.

## Step 1. Locate the packaged benchmark data

This cell resolves where the benchmark data lives before any runtime logic starts.

| Input | Purpose |
| --- | --- |
| Default Kaggle dataset root | Used during the real Kaggle notebook execution |
| `RULESHIFT_DATA_ROOT` | Preferred override for local validation or automated smoke tests |
| Legacy `RULESHIFT_RUNTIME_ROOT` | Backward-compatible fallback for older packaged layouts |

The notebook prefers the new minimal `data/` layout, but it can still read the older `src/frozen_splits/` layout when needed. If no valid directory is found, it stops immediately with a clear error instead of failing later in a more confusing way.

In [ ]:
import os
from pathlib import Path

import kaggle_benchmarks as kbench

DEFAULT_RUNTIME_ROOT = Path("/kaggle/input/datasets/raptorengineer/ruleshift-runtime")
LEGACY_RUNTIME_ROOT = Path(os.environ.get("RULESHIFT_RUNTIME_ROOT", DEFAULT_RUNTIME_ROOT / "src"))
DATA_ROOT = Path(os.environ.get("RULESHIFT_DATA_ROOT", DEFAULT_RUNTIME_ROOT / "data"))
if not DATA_ROOT.is_dir() and (LEGACY_RUNTIME_ROOT / "frozen_splits").is_dir():
    DATA_ROOT = LEGACY_RUNTIME_ROOT / "frozen_splits"
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"Runtime dataset not found at {DATA_ROOT} -- "
        "attach the raptorengineer/ruleshift-runtime dataset."
    )

## Step 2. Define the benchmark building blocks

The runtime is intentionally split into smaller sections so each concern can be understood on its own.

| Runtime section | What it contains |
| --- | --- |
| Shared constants and types | Labels, response schema, filenames, and benchmark-wide settings |
| Response normalization | Logic that converts different model outputs into the expected labels |
| Scoring helpers | Functions that compare predictions against the frozen targets |
| Dataset loading | Functions that read the public and private benchmark rows |

```mermaid
flowchart LR
    A["Constants and paths"] --> B["Response schema"]
    B --> C["Normalization helpers"]
    C --> D["Scoring helpers"]
    D --> E["Dataset loaders"]
```

This first code block defines the shared vocabulary that every later cell depends on.

In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import Final

class Label(str, Enum):
    type_a = "type_a"
    type_b = "type_b"

PROBE_COUNT: Final[int] = 4
PRIVATE_DATASET_ROOT_ENV_VAR: Final[str] = "RULESHIFT_PRIVATE_DATASET_ROOT"

_PUBLIC_ROWS_FILENAME: Final[str] = "public_leaderboard_rows.json"
_PRIVATE_ROWS_FILENAME: Final[str] = "private_leaderboard_rows.json"
_ALLOWED_LABELS: Final[frozenset[str]] = frozenset(label.value for label in Label)
_PROBE_FIELDS: Final[tuple[str, ...]] = ("probe_6", "probe_7", "probe_8", "probe_9")

@dataclass(frozen=True)
class BinaryResponse:
    probe_6: Label
    probe_7: Label
    probe_8: Label
    probe_9: Label

    def as_tuple(self) -> tuple[str, str, str, str]:
        return (
            _coerce_label(self.probe_6, "probe_6"),
            _coerce_label(self.probe_7, "probe_7"),
            _coerce_label(self.probe_8, "probe_8"),
            _coerce_label(self.probe_9, "probe_9"),
        )

class KaggleExecutionError(RuntimeError):
    pass

## Step 3. Normalize model responses

Model outputs are not always shaped the same way, so this section standardizes them before scoring.

| Accepted shape | Example |
| --- | --- |
| Structured response object | `BinaryResponse(...)` |
| Dictionary-like response | `{"probe_6": "type_a", ...}` |
| Plain text response | `type_a, type_b, type_a, type_b` |

The helpers are forgiving about format, but strict about labels: each normalized answer must end up as exactly four values drawn from `type_a` and `type_b`.

In [ ]:
def normalize_binary_response(response: object) -> tuple[str, ...] | None:
    if response is None:
        return None
    if isinstance(response, BinaryResponse):
        return response.as_tuple()
    if isinstance(response, str):
        return _parse_text_response(response)
    br = _try_coerce(response)
    return br.as_tuple() if br is not None else None

def _parse_text_response(text: str) -> tuple[str, ...] | None:
    tokens = tuple(
        t.strip().lower()
        for t in text.strip().strip("`").replace("\n", ",").split(",")
        if t.strip()
    )
    return _norm_labels(tokens)

def _try_coerce(response: object) -> BinaryResponse | None:
    if isinstance(response, dict):
        vals = response
    elif hasattr(response, "__getitem__") and hasattr(response, "keys"):
        try:
            vals = {f: response[f] for f in _PROBE_FIELDS}
        except (KeyError, TypeError):
            return None
    elif all(hasattr(response, f) for f in _PROBE_FIELDS):
        vals = {f: getattr(response, f) for f in _PROBE_FIELDS}
    else:
        return None
    try:
        labels = tuple(Label(_coerce_label(vals[f], f)) for f in _PROBE_FIELDS)
    except (KeyError, TypeError):
        return None
    return BinaryResponse(*labels)

def _coerce_label(value: object, field: str) -> str:
    try:
        return _normalize_label(value)
    except ValueError as exc:
        raise ValueError(f"invalid field {field}: {value!r}") from exc

def _norm_labels(
    labels: tuple[str, ...] | tuple[Label, ...] | None,
) -> tuple[str, ...] | None:
    if labels is None:
        return None
    try:
        result = tuple(
            _normalize_label(lbl.value if isinstance(lbl, Label) else lbl)
            for lbl in labels
        )
    except ValueError:
        return None
    return result if len(result) == PROBE_COUNT else None

def _normalize_label(value: object) -> str:
    if isinstance(value, Enum):
        value = value.value
    elif hasattr(value, "value"):
        value = value.value
    normalized = str(value).strip().lower()
    if normalized not in _ALLOWED_LABELS:
        raise ValueError(f"unknown label: {value}")
    return normalized

## Step 4. Score each benchmark episode

This section converts one benchmark prompt into one measurable score.

| Stage | Result |
| --- | --- |
| Prompt the model | Request a structured four-probe answer |
| Normalize the response | Convert the returned value into canonical labels |
| Compare with targets | Count correct probe predictions |
| Return `(numerator, denominator)` | Produce the episode contribution to the final benchmark score |

If the model call fails or returns something unscoreable, the runtime raises an explicit error so the failure is easy to diagnose.

In [ ]:
def score_episode(
    predictions: tuple[str, ...] | tuple[Label, ...] | None,
    targets: tuple[str, ...] | tuple[Label, ...],
) -> tuple[int, int]:
    norm_targets = _norm_labels(targets)
    if norm_targets is None:
        raise ValueError(f"targets must contain exactly {PROBE_COUNT} valid labels")
    norm_preds = _norm_labels(predictions)
    if norm_preds is None:
        return (0, PROBE_COUNT)
    return (
        sum(p == t for p, t in zip(norm_preds, norm_targets)),
        PROBE_COUNT,
    )

def run_binary_task(
    *,
    llm: object,
    prompt_binary: str,
    probe_targets: tuple[str, ...] | tuple[Label, ...],
) -> tuple[int, int]:
    try:
        response = llm.prompt(prompt_binary, schema=BinaryResponse)
    except Exception as exc:
        raise KaggleExecutionError("llm.prompt failed") from exc

    try:
        normalized = normalize_binary_response(response)
    except ValueError as exc:
        raise KaggleExecutionError(f"invalid binary response: {exc}") from exc

    if normalized is None:
        raise KaggleExecutionError(
            f"unscoreable response of type {type(response).__name__}"
        )
    return score_episode(normalized, probe_targets)

## Step 5. Load the frozen benchmark rows

This part materializes the full evaluation set that the task will iterate over.

| Split | Source | When it is used |
| --- | --- | --- |
| Public rows | Packaged notebook dataset | Always |
| Private rows | `RULESHIFT_PRIVATE_DATASET_ROOT` | Only when the private dataset is available |

At the end of the cell, `leaderboard_rows` becomes the single source of truth for the benchmark loop. If no rows are available, the notebook stops immediately.

In [ ]:
import json

def load_public_rows() -> list[dict[str, object]]:
    return _load_rows(DATA_ROOT / _PUBLIC_ROWS_FILENAME)

def load_private_rows(
    private_dataset_root: Path | str | None = None,
) -> list[dict[str, object]]:
    if private_dataset_root is None:
        env = os.environ.get(PRIVATE_DATASET_ROOT_ENV_VAR)
        if not env:
            raise FileNotFoundError(
                f"Private dataset not found. Set {PRIVATE_DATASET_ROOT_ENV_VAR} or "
                "pass private_dataset_root explicitly."
            )
        private_dataset_root = env
    root = Path(private_dataset_root)
    return _load_rows(root / _PRIVATE_ROWS_FILENAME)

def _load_rows(path: Path) -> list[dict[str, object]]:
    rows = json.loads(path.read_text("utf-8"))
    return [
        {
            "episode_id": row["episode_id"],
            "split": row["split"],
            "prompt_binary": row["prompt_binary"],
            "probe_targets": _normalize_probe_targets(row["probe_targets"]),
        }
        for row in rows
    ]

def _normalize_probe_targets(values: object) -> tuple[str, ...]:
    if not isinstance(values, list | tuple):
        raise ValueError("probe_targets must be a list or tuple")
    labels = _norm_labels(tuple(values))
    if labels is None:
        raise ValueError(f"probe_targets must contain exactly {PROBE_COUNT} valid labels")
    return labels

leaderboard_rows = load_public_rows()
PRIVATE_DATASET_ROOT = os.environ.get(PRIVATE_DATASET_ROOT_ENV_VAR)
if PRIVATE_DATASET_ROOT:
    leaderboard_rows.extend(load_private_rows(PRIVATE_DATASET_ROOT))

if not leaderboard_rows:
    raise RuntimeError("No benchmark episodes were loaded.")

## Official task

`ruleshift_benchmark_v1_binary` is the leaderboard entry point.

| Task contract | Value |
| --- | --- |
| Input | A Kaggle Benchmarks-compatible `llm` object |
| Work performed | Evaluate every loaded benchmark row |
| Output | `(numerator, denominator)` |
| Side effect | Stores a readable summary in `_RULESHIFT_RESULT` |

In short, this task is the official wrapper around the runtime and scoring pipeline defined above.

## Step 6. Register the Kaggle task

This cell turns the scoring loop into a Kaggle Benchmarks task.

Execution sequence:

1. Iterate through every row in `leaderboard_rows`.
2. Send the prompt to the provided `llm`.
3. Score the four probe predictions for that episode.
4. Aggregate the episode totals into a final benchmark result.
5. Save a compact summary so the notebook can display it after the run.

This is the main evaluation cell: everything above prepares data and helpers for this loop.

In [ ]:
_RULESHIFT_RESULT = None


@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of marker records, "
        "then predict four post-shift probe outputs."
    ),
)
def ruleshift_benchmark_v1_binary(llm) -> tuple[int, int]:
    import json

    global _RULESHIFT_RESULT

    numerator = 0
    denominator = 0
    for row in leaderboard_rows:
        num_correct, total = run_binary_task(
            llm=llm,
            prompt_binary=row["prompt_binary"],
            probe_targets=row["probe_targets"],
        )
        numerator += int(num_correct)
        denominator += int(total)

    if denominator == 0:
        raise RuntimeError("evaluation produced a zero denominator")

    _RULESHIFT_RESULT = {
        "score": numerator / denominator,
        "numerator": numerator,
        "denominator": denominator,
        "episodes": len(leaderboard_rows),
    }

    print(json.dumps(_RULESHIFT_RESULT, indent=2))
    return (numerator, denominator)

## Step 7. Run a notebook smoke check

This cell executes the registered task once inside the notebook and displays the resulting summary.

| Why run this cell? | What you learn |
| --- | --- |
| Sanity check | Confirms the notebook can run end to end |
| Quick inspection | Shows score, numerator, denominator, and episode count |
| Debugging help | Surfaces runtime issues before submission |

If this smoke check succeeds, the notebook is in a good state for Kaggle task selection.

In [ ]:
score = ruleshift_benchmark_v1_binary(kbench.llm)
result = _RULESHIFT_RESULT
if result is None:
    raise RuntimeError("ruleshift_benchmark_v1_binary did not populate _RULESHIFT_RESULT")

result

## Task selection

The final step is to tell Kaggle which registered task name should be used as the notebook entry point.

## Step 8. Mark the official entry point

This final cell binds the notebook to the official task name.

> Expected outcome: Kaggle will use `ruleshift_benchmark_v1_binary` as the callable benchmark entry when the notebook is evaluated.

In [ ]:
%choose ruleshift_benchmark_v1_binary